# KVO_06 — Automatic SAM3 crown discovery on Baego

**Question:** can official Meta SAM3 automatically discover tree-crown objects in Baego RGB imagery, and how often does it recover the high-confidence crowns in `Crown_Nadine.shp`?

`Crown_Nadine` is treated as a **high-confidence positive validation set, not an exhaustive crown census**. Therefore:
- recovery/recall of Nadine crowns can be measured;
- matched-object IoU/Dice can be measured;
- unmatched SAM3 objects are **unverified**, not false positives;
- conventional whole-area precision is deliberately not reported.

The notebook runs SAM3 **without Nadine-derived boxes**. It tiles the spatial envelope covered by the Nadine reference sample, uses the text prompt `tree crown`, merges duplicate detections from overlapping tiles, and matches the resulting automatic crown objects to the manual reference crowns. This is a discovery benchmark, complementary to KVO_05's box-prompt delineation benchmark.


## 0 — Host environment and configuration

Use a **GPU runtime**. KVO_06 reuses the same isolated official SAM3 setup as KVO_05.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil, math
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape, box
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_SAM3_Baego_Crown_Discovery"
TILES=OUT/"tiles"; RAW=OUT/"sam3_raw"; VIS=OUT/"examples"
for p in [OUT,TILES,RAW,VIS]: p.mkdir(parents=True,exist_ok=True)

RASTER=GEO/"Baego_georef.tif"
ANNOT=AD/"Crown_Nadine.shp"
assert RASTER.exists() and ANNOT.exists()

MIN_REF_AREA_M2=5.0
PROMPT="tree crown"
CONFIDENCE=0.35
TILE_PX=1008
OVERLAP_FRAC=0.25
REFERENCE_BUFFER_M=30.0
MIN_PRED_AREA_M2=2.0
MAX_PRED_AREA_M2=500.0
DUP_IOU=0.50
MATCH_IOU_THRESHOLDS=[0.25,0.50,0.70]
FORCE_TILES=False
FORCE_SAM=False
print("OUT:",OUT)


## 1 — Define the discovery footprint and make overlapping RGB tiles

The test footprint is the bounding envelope of the high-confidence Nadine sample plus a 30 m buffer. **Individual Nadine crowns are not used as prompts and do not determine tile centres.** SAM3 sees ordinary overlapping image tiles.

If the Nadine polygons span nearly the entire Baego mosaic, the cell reports the resulting tile count before inference so the workload is explicit.


In [ ]:
with rasterio.open(RASTER) as src:
    crs=src.crs; rb=box(*src.bounds); transform=src.transform
g=gpd.read_file(ANNOT)
if g.crs is None: raise ValueError("Crown_Nadine.shp has no CRS")
if g.crs!=crs: g=g.to_crs(crs)
g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
g=g[g.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()
g["crown_area_m2"]=g.geometry.area
g=g[g.crown_area_m2>=MIN_REF_AREA_M2].copy().reset_index(drop=True)
g["ref_id"]=np.arange(len(g),dtype=int)

foot=box(*g.total_bounds).buffer(REFERENCE_BUFFER_M).intersection(rb)
print("High-confidence reference crowns:",len(g))
print("Reference area median m²:",float(g.crown_area_m2.median()))
print("Discovery footprint area ha:",foot.area/10000)

with rasterio.open(RASTER) as src:
    fw=rasterio.windows.from_bounds(*foot.bounds,transform=src.transform).round_offsets().round_lengths()
    cmin=max(0,int(fw.col_off)); rmin=max(0,int(fw.row_off))
    cmax=min(src.width,int(fw.col_off+fw.width)); rmax=min(src.height,int(fw.row_off+fw.height))
    step=max(1,int(round(TILE_PX*(1-OVERLAP_FRAC))))
    cols=list(range(cmin,max(cmin+1,cmax-TILE_PX+1),step))
    rows=list(range(rmin,max(rmin+1,rmax-TILE_PX+1),step))
    if not cols or cols[-1]+TILE_PX<cmax: cols.append(max(cmin,cmax-TILE_PX))
    if not rows or rows[-1]+TILE_PX<rmax: rows.append(max(rmin,rmax-TILE_PX))
    rec=[]
    for r0 in rows:
        for c0 in cols:
            w=Window(c0,r0,min(TILE_PX,src.width-c0),min(TILE_PX,src.height-r0))
            wb=box(*rasterio.windows.bounds(w,src.transform))
            if not wb.intersects(foot): continue
            tid=f"r{r0:06d}_c{c0:06d}"
            p=TILES/f"{tid}.jpg"
            if FORCE_TILES or not p.exists():
                rgb=np.moveaxis(src.read([1,2,3],window=w),0,-1).astype(np.uint8)
                Image.fromarray(rgb).save(p,quality=95)
            rec.append(dict(tile_id=tid,path=str(p),row_off=int(r0),col_off=int(c0),
                            height=int(w.height),width=int(w.width)))
tiles=pd.DataFrame(rec).drop_duplicates("tile_id").reset_index(drop=True)
tiles.to_csv(OUT/"tile_manifest.csv",index=False)
print("Discovery tiles:",len(tiles),f"(overlap {OVERLAP_FRAC:.0%})")
display(tiles.head())


## 2 — Isolated official SAM3 environment

In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

In [ ]:
from google.colab import userdata
try: HF_TOKEN=userdata.get("HF_TOKEN")
except Exception: HF_TOKEN=None
assert HF_TOKEN,"Add your Hugging Face read token to Colab Secrets as HF_TOKEN."
print("HF_TOKEN found (value not displayed).")


## 4 — Automatic SAM3 text-prompt discovery

Each image tile is encoded once and queried with the text prompt **tree crown**. No manual crown boxes or locations are supplied. Raw masks, boxes and scores are persisted per tile and skipped on reruns.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import os,sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); prompt=sys.argv[3]
threshold=float(sys.argv[4]); force=sys.argv[5]=="1"
outdir.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__()
torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe)
processor=Sam3Processor(model,confidence_threshold=threshold)
print("SAM3 ready; prompt=",prompt," threshold=",threshold)
for r in tqdm(df.itertuples(),total=len(df),desc="DISCOVER"):
    op=outdir/f"{r.tile_id}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    out=processor.set_text_prompt(prompt=prompt,state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    boxes=out["boxes"].detach().float().cpu().numpy()
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,boxes=boxes,scores=scores)
print("DONE")
''')
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(OUT/"tile_manifest.csv"),str(RAW),PROMPT,str(CONFIDENCE),"1" if FORCE_SAM else "0"]
p=subprocess.run(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(p.stdout)
if p.returncode!=0: raise RuntimeError("SAM3 discovery subprocess failed; error printed above.")


## 5 — Convert masks to georeferenced crown polygons

Small/implausibly large objects are filtered by physical area. Objects are clipped to the discovery footprint. This step does not use Crown_Nadine.


In [ ]:
pred=[]
with rasterio.open(RASTER) as src:
    pixarea=abs(src.transform.a*src.transform.e)
    for r in tqdm(tiles.itertuples(),total=len(tiles),desc="POLYGONIZE"):
        z=np.load(RAW/f"{r.tile_id}.npz")
        masks=z["masks"]; scores=z["scores"]
        w=Window(r.col_off,r.row_off,r.width,r.height); tr=src.window_transform(w)
        for j,(m,score) in enumerate(zip(masks,scores)):
            if m.shape!=(r.height,r.width):
                m=np.asarray(Image.fromarray(m).resize((r.width,r.height),Image.Resampling.NEAREST))
            if not m.any(): continue
            geoms=[shape(gg) for gg,val in shapes(m.astype(np.uint8),mask=m.astype(bool),transform=tr) if val==1]
            if not geoms: continue
            geom=max(geoms,key=lambda x:x.area).intersection(foot)
            if geom.is_empty: continue
            area=geom.area
            if not (MIN_PRED_AREA_M2<=area<=MAX_PRED_AREA_M2): continue
            pred.append(dict(tile_id=r.tile_id,local_id=j,score=float(score),area_m2=float(area),geometry=geom))
pg=gpd.GeoDataFrame(pred,crs=crs)
print("Raw plausible crown objects:",len(pg))


## 6 — Remove duplicate crowns from overlapping tiles

Greedy score-ordered spatial NMS removes near-duplicate masks with polygon IoU ≥ `DUP_IOU`. It does not use the manual reference crowns.


In [ ]:
def geom_iou(a,b):
    inter=a.intersection(b).area
    if inter<=0: return 0.0
    u=a.area+b.area-inter
    return inter/u if u>0 else 0.0

pg=pg.sort_values("score",ascending=False).reset_index(drop=True)
keep=[]; kept_geoms=[]
for i,r in tqdm(pg.iterrows(),total=len(pg),desc="DEDUP"):
    cand=r.geometry
    duplicate=False
    # bounding-box prefilter against already kept objects
    for kg in kept_geoms:
        if not cand.intersects(kg): continue
        if geom_iou(cand,kg)>=DUP_IOU:
            duplicate=True; break
    if not duplicate:
        keep.append(i); kept_geoms.append(cand)
crowns=pg.loc[keep].copy().reset_index(drop=True)
crowns["crown_id"]=np.arange(len(crowns),dtype=int)
crowns.to_file(OUT/"Baego_SAM3_discovered_crowns.gpkg",layer="crowns",driver="GPKG")
print("Unique discovered crowns:",len(crowns))
print("Density within discovery footprint:",len(crowns)/(foot.area/10000),"crowns/ha")
display(crowns[["crown_id","score","area_m2"]].describe())


## 7 — Match automatic crowns to Crown_Nadine

This is **positive-reference recovery**, not a precision test. Each Nadine crown is matched to the automatic SAM3 object with the highest polygon IoU. Automatic objects without a Nadine match remain **unverified**.


In [ ]:
sidx=crowns.sindex
matches=[]
for r in tqdm(g.itertuples(),total=len(g),desc="MATCH REFERENCES"):
    cand_idx=list(sidx.query(r.geometry,predicate="intersects"))
    best_i=-1; best_iou=0.0
    for i in cand_idx:
        v=geom_iou(r.geometry,crowns.geometry.iloc[i])
        if v>best_iou: best_iou=v; best_i=int(i)
    if best_i>=0:
        p=crowns.iloc[best_i]
        inter=r.geometry.intersection(p.geometry).area
        dice=2*inter/(r.geometry.area+p.geometry.area)
        cd=r.geometry.centroid.distance(p.geometry.centroid)
        score=float(p.score); pred_id=int(p.crown_id); pred_area=float(p.area_m2)
    else:
        dice=0.0; cd=np.nan; score=np.nan; pred_id=-1; pred_area=np.nan
    matches.append(dict(ref_id=int(r.ref_id),reference_area_m2=float(r.crown_area_m2),
                        pred_crown_id=pred_id,best_iou=best_iou,dice=dice,
                        centroid_displacement_m=cd,sam_score=score,pred_area_m2=pred_area))
match=pd.DataFrame(matches)
match.to_csv(OUT/"Crown_Nadine_discovery_matches.csv",index=False)

summary={"n_reference":len(match),"n_discovered":len(crowns)}
for t in MATCH_IOU_THRESHOLDS:
    summary[f"recovered_iou_ge_{t:.2f}"]=int((match.best_iou>=t).sum())
    summary[f"recall_iou_ge_{t:.2f}"]=float((match.best_iou>=t).mean())
summary["median_best_iou"]=float(match.best_iou.median())
summary["median_iou_matched_ge_025"]=float(match.loc[match.best_iou>=.25,"best_iou"].median())
summary["median_dice_matched_ge_025"]=float(match.loc[match.best_iou>=.25,"dice"].median())
display(pd.DataFrame([summary]).T.rename(columns={0:"value"}))
(OUT/"discovery_summary.json").write_text(json.dumps(summary,indent=2))


## 8 — Diagnostic plots and interpretation

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
ax.hist(match.best_iou,bins=np.linspace(0,1,21))
for t in MATCH_IOU_THRESHOLDS: ax.axvline(t,linestyle="--",linewidth=1)
ax.set_xlabel("Best IoU with an automatic SAM3 crown"); ax.set_ylabel("Crown_Nadine references")
ax.set_title("Automatic SAM3 recovery of high-confidence crowns")
fig.tight_layout(); fig.savefig(OUT/"reference_recovery_iou.png",dpi=180); plt.show()

fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(match.reference_area_m2,match.best_iou,s=14,alpha=.5)
ax.set_xscale("log"); ax.set_ylim(-.02,1.02)
ax.set_xlabel("Crown_Nadine area (m², log scale)"); ax.set_ylabel("Best automatic-crown IoU")
ax.set_title("Discovery performance vs reference crown size")
fig.tight_layout(); fig.savefig(OUT/"recovery_vs_crown_area.png",dpi=180); plt.show()

print("IMPORTANT: unmatched SAM3 objects are UNVERIFIED, not false positives.")
print("Do not report whole-area precision from Crown_Nadine because the reference is intentionally non-exhaustive.")


## 9 — Decision gate

Interpret KVO_05 and KVO_06 together:

- **KVO_05:** if a high-confidence crown is approximately located, how accurately can SAM3 delineate its boundary?
- **KVO_06:** without manual crown locations, how often does SAM3 independently recover those high-confidence crowns?

If discovery recall is useful, the next step is to intersect the automatic crown objects with the ROI-masked KVO_04 Musanga probability raster and derive crown-level Musanga probabilities. If text-prompt discovery is weak, retain SAM3 as a delineator and seed candidate locations from KVO_04 probability peaks instead.
